In [4]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway

df = pd.read_csv('pollution_processed.csv')

In [5]:
# Q1 - Correlation
print("=== Q1: CORRELATION ===")
corr, p = stats.pearsonr(df['Avg_Industrial_Emission'], df['Avg_AQI'])
print("Correlation:", round(corr, 4))
print("P-Value    :", round(p, 4))
print("Result     : Weak — multiple factors cause AQI")

print()

=== Q1: CORRELATION ===
Correlation: -0.0054
P-Value    : 0.8219
Result     : Weak — multiple factors cause AQI



In [6]:
# Q2 - Hypothesis Test
print("=== Q2: HYPOTHESIS TEST ===")
high = df[df['Avg_Public_Transport'] > 50]['Avg_AQI']
low  = df[df['Avg_Public_Transport'] <= 50]['Avg_AQI']
t, p = stats.ttest_ind(high, low)
print("High Transport AQI:", round(high.mean(), 2))
print("Low Transport AQI :", round(low.mean(), 2))
print("P-Value           :", round(p, 6))
print("Result            : Public transport reduces AQI!" if p < 0.05 else "Not proven")

print()

=== Q2: HYPOTHESIS TEST ===
High Transport AQI: 157.69
Low Transport AQI : 169.43
P-Value           : 0.001767
Result            : Public transport reduces AQI!



In [7]:
# Q3 - Regression
print("=== Q3: REGRESSION ===")
slope, intercept, r, p, se = stats.linregress(
    df['Avg_Vehicular_Emission'], df['Avg_AQI']
)
print("R-Squared:", round(r**2, 4))
print("Formula  : AQI =", round(slope, 6), "x Vehicles +", round(intercept, 2))

print()

=== Q3: REGRESSION ===
R-Squared: 0.0001
Formula  : AQI = 0.0 x Vehicles + 166.07



In [9]:
# Q4: Moving Averages
print("\n" + "=" * 50)
print("Q4: MOVING AVERAGES (Yearly AQI)")
print("=" * 50)
yearly = df.groupby('Year')['Avg_AQI'].mean().reset_index()
yearly.columns = ['Year', 'Avg_AQI']
yearly['Avg_AQI'] = yearly['Avg_AQI'].round(2)
yearly['MA_3yr']  = yearly['Avg_AQI'].rolling(window=3, min_periods=1).mean().round(2)
yearly['MA_5yr']  = yearly['Avg_AQI'].rolling(window=5, min_periods=1).mean().round(2)
print(yearly.to_string(index=False))


Q4: MOVING AVERAGES (Yearly AQI)
 Year  Avg_AQI  MA_3yr  MA_5yr
 2010   154.01  154.01  154.01
 2011   156.13  155.07  155.07
 2012   157.29  155.81  155.81
 2013   158.66  157.36  156.52
 2014   161.49  159.15  157.52
 2015   176.89  165.68  162.09
 2016   177.40  171.93  166.35
 2017   176.61  176.97  170.21
 2018   173.18  175.73  173.11
 2019   152.71  167.50  171.36
 2020   131.40  152.43  162.26
 2021   174.57  152.89  161.69
 2022   174.55  160.17  161.28
 2023   177.85  175.66  162.22
 2024   180.16  177.52  167.71
 2025   182.04  180.02  177.83


In [13]:
# Q5 - Chi Square
print("=== Q5: CHI SQUARE ===")
df['AQI_Improved'] = df['Avg_AQI_Improvement'].apply(
    lambda x: 'Yes' if x < 0 else 'No'
)
table = pd.crosstab(df['NCAP_Flag'], df['AQI_Improved'])
chi2, p, dof, exp = chi2_contingency(table)
print(table)
print("Chi-Square:", round(chi2, 4))
print("P-Value   :", round(p, 6))
print("Result    : NCAP policy works!" if p < 0.05 else "Not significant")

=== Q5: CHI SQUARE ===
AQI_Improved   No  Yes
NCAP_Flag             
No            513  557
Yes           365  315
Chi-Square: 5.2384
P-Value   : 0.022093
Result    : NCAP policy works!


In [14]:
# Q6 - Time Series
print("=== Q6: TIME SERIES ===")
monthly = df.groupby(['Year', 'Month'])['Avg_AQI'].mean().reset_index()
monthly['Date'] = pd.to_datetime(
    monthly['Year'].astype(str) + '-' +
    monthly['Month'].astype(int).astype(str) + '-01'
)
monthly = monthly.set_index('Date').sort_index()
print("Total months:", len(monthly))
print("Highest AQI month:", monthly['Avg_AQI'].idxmax().strftime('%B %Y'),
      "—", round(monthly['Avg_AQI'].max(), 2))
print("Lowest AQI month :", monthly['Avg_AQI'].idxmin().strftime('%B %Y'),
      "—", round(monthly['Avg_AQI'].min(), 2))

print()

=== Q6: TIME SERIES ===
Total months: 102
Highest AQI month: November 2017 — 350.03
Lowest AQI month : July 2018 — 73.81



In [15]:
# Q7 - ANOVA
print("=== Q7: ANOVA ===")
groups = [g['Avg_AQI'].values for n, g in df.groupby('State')]
f, p = f_oneway(*groups)
print("F-Statistic:", round(f, 4))
print("P-Value    :", round(p, 6))
print("Result     : States genuinely different!" if p < 0.05 else "No difference")

print()

=== Q7: ANOVA ===
F-Statistic: 49.2349
P-Value    : 0.0
Result     : States genuinely different!



In [16]:
# Q8 - Confidence Interval
print("=== Q8: CONFIDENCE INTERVAL ===")
data = df['Avg_Water_Contamination']
mean = data.mean()
se   = data.std() / np.sqrt(len(data))
ci   = stats.t.interval(0.95, df=len(data)-1, loc=mean, scale=se)
print("Mean  :", round(mean, 4))
print("95% CI:", round(ci[0], 4), "to", round(ci[1], 4))

print()

=== Q8: CONFIDENCE INTERVAL ===
Mean  : 1.8335
95% CI: 1.7785 to 1.8886



In [17]:
# Q9 - Sensitivity Analysis
print("=== Q9: SENSITIVITY ANALYSIS ===")
base = df['Avg_AQI'].mean()
print("Current AQI:", round(base, 2))
for pct in [10, 20, 30, 40, 50]:
    new  = base * (1 - pct/100 * 0.30)
    drop = base - new
    print(f"{pct}% cut → AQI: {new:.2f} (drop: {drop:.2f})")

print()

=== Q9: SENSITIVITY ANALYSIS ===
Current AQI: 166.31
10% cut → AQI: 161.32 (drop: 4.99)
20% cut → AQI: 156.33 (drop: 9.98)
30% cut → AQI: 151.34 (drop: 14.97)
40% cut → AQI: 146.35 (drop: 19.96)
50% cut → AQI: 141.37 (drop: 24.95)



In [18]:
# Q10 - Export
print("=== Q10: EXPORT FOR POWER BI ===")
export = df.groupby(['State', 'City', 'Year']).agg(
    Avg_AQI      = ('Avg_AQI', 'mean'),
    Std_AQI      = ('Avg_AQI', 'std'),
    Avg_Water    = ('Avg_Water_Contamination', 'mean'),
    Avg_Emission = ('Avg_Industrial_Emission', 'mean')
).reset_index().round(4)
export.to_csv('datascience_stats_for_powerbi.csv', index=False)
print("Rows exported:", len(export))
print("File saved: datascience_stats_for_powerbi.csv")

=== Q10: EXPORT FOR POWER BI ===
Rows exported: 416
File saved: datascience_stats_for_powerbi.csv
